<a href="https://colab.research.google.com/github/usharani-192312304/ITA0605-ML/blob/main/Exp09_Linear_vs_Polynomial_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 9: Linear vs Polynomial Regression
**Course:** ITA06 - Machine Learning (Lab)

## Aim
Compare Linear and Polynomial Regression using Python.

## Background
Linear regression can only fit straight lines. **Polynomial regression** keeps the same linear-in-parameters model but adds powers of the input (`x, x^2, x^3, ...`) so it can fit curves. A higher degree lowers the training error but eventually **overfits** (high variance); too low a degree **underfits** (high bias). We choose the degree using validation performance.

## Pipeline followed in this notebook
1. Step 1: Data Collection
2. Step 2: Data Loading
3. Step 3: Data Preprocessing
4. Step 4: Exploratory Data Analysis (EDA)
5. Step 5: Model Training
6. Step 6: Model Evaluation
7. Step 7: Prediction on New Data

> **How to run:** Open in Google Colab -> *Runtime -> Run all*. No files need to be uploaded; the data is created or loaded inside the notebook.

## Setup: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
print("Libraries imported successfully")

## Step 1: Data Collection
We simulate a **position level vs salary** relationship that is clearly *non-linear* (salary rises faster at senior levels), with noise. Saved as `position_salary.csv`.

In [ ]:
rng = np.random.RandomState(42)
n = 50
level = np.round(rng.uniform(1, 10, n), 2)
salary = (20000 + 3000 * level + 1500 * level ** 2 + rng.normal(0, 9000, n)).round(0)
pd.DataFrame({"Level": level, "Salary": salary}).to_csv("position_salary.csv", index=False)
print("Saved position_salary.csv")

## Step 2: Data Loading

In [ ]:
df = pd.read_csv("position_salary.csv").sort_values("Level").reset_index(drop=True)
df.head()

## Step 3: Data Preprocessing
Check data quality, then split into train and test sets. Scaling is done inside the model pipeline because high powers of `x` get huge.

In [ ]:
from sklearn.model_selection import train_test_split

print("Missing values:", df.isnull().sum().sum())
X = df[["Level"]]
y = df["Salary"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
print(df.describe().T)
print("Pearson correlation:", df.corr().loc["Level", "Salary"].round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(data=df, x="Level", y="Salary", ax=axes[0]); axes[0].set_title("The relationship looks curved")
sns.histplot(df["Salary"], kde=True, ax=axes[1]); axes[1].set_title("Salary distribution (right-skewed)")
plt.tight_layout(); plt.show()

## Step 5: Model Training
Train a linear model and polynomial models of increasing degree using a pipeline: `PolynomialFeatures -> StandardScaler -> LinearRegression`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression

def make_model(degree):
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())

models = {d: make_model(d).fit(X_train, y_train) for d in range(1, 11)}
print("Trained polynomial models of degree 1 to 10 (degree 1 = ordinary linear regression)")

## Step 6: Model Evaluation
Compare train/test RMSE and R-squared for every degree, use **5-fold cross-validation** to pick the best degree, and plot the fits.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

rows = []
for d, m in models.items():
    rows.append({"Degree": d,
                 "Train RMSE": np.sqrt(mean_squared_error(y_train, m.predict(X_train))),
                 "Test RMSE": np.sqrt(mean_squared_error(y_test, m.predict(X_test))),
                 "Train R2": r2_score(y_train, m.predict(X_train)),
                 "Test R2": r2_score(y_test, m.predict(X_test)),
                 "CV R2": cross_val_score(make_model(d), X_train, y_train, cv=5, scoring="r2").mean()})
results = pd.DataFrame(rows).set_index("Degree").round(3)
print(results)
# parsimony rule: simplest degree whose CV score is within 0.005 of the best
best_cv = results["CV R2"].max()
best_degree = int(results.index[results["CV R2"] >= best_cv - 0.005][0])
print("\nHighest CV R2 =", best_cv, "| simplest degree within 0.005 of it:", best_degree)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
axes[0].plot(results.index, results["Train RMSE"], marker="o", label="Train RMSE")
axes[0].plot(results.index, results["Test RMSE"], marker="s", label="Test RMSE")
axes[0].axvline(best_degree, color="g", linestyle="--", label=f"best (CV) = {best_degree}")
axes[0].set_xlabel("Polynomial degree"); axes[0].set_ylabel("RMSE"); axes[0].set_title("Bias-variance trade-off"); axes[0].legend()

grid = np.linspace(1, 10, 300).reshape(-1, 1)
axes[1].scatter(X_train, y_train, alpha=.6, label="Train"); axes[1].scatter(X_test, y_test, color="orange", label="Test")
for d, style in [(1, "r--"), (best_degree, "g-"), (10, "m:")]:
    axes[1].plot(grid, models[d].predict(pd.DataFrame(grid, columns=["Level"])), style, label=f"degree {d}", linewidth=2)
axes[1].set_xlabel("Level"); axes[1].set_ylabel("Salary"); axes[1].set_title("Underfit vs good fit vs overfit"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
lin, poly = models[1], models[best_degree]
comparison = pd.DataFrame({
    "Linear (deg 1)": [r2_score(y_test, lin.predict(X_test)), np.sqrt(mean_squared_error(y_test, lin.predict(X_test)))],
    f"Polynomial (deg {best_degree})": [r2_score(y_test, poly.predict(X_test)), np.sqrt(mean_squared_error(y_test, poly.predict(X_test)))],
}, index=["Test R2", "Test RMSE"]).round(3)
print(comparison)

res_lin = y_test - lin.predict(X_test)
res_poly = y_test - poly.predict(X_test)
plt.scatter(X_test, res_lin, label="Linear residuals"); plt.scatter(X_test, res_poly, label="Polynomial residuals")
plt.axhline(0, color="k", linestyle="--"); plt.xlabel("Level"); plt.ylabel("Residual"); plt.legend(); plt.title("Residual comparison"); plt.show()

## Step 7: Prediction on New Data
Predict salary for a few new levels with both models.

In [ ]:
new_levels = pd.DataFrame({"Level": [2.5, 6.5, 9.5]})
out = new_levels.copy()
out["Linear"] = lin.predict(new_levels).round(0)
out[f"Polynomial (deg {best_degree})"] = poly.predict(new_levels).round(0)
out["True trend (no noise)"] = (20000 + 3000 * new_levels["Level"] + 1500 * new_levels["Level"] ** 2).round(0)
out

## Conclusion
The straight line underfits the curved data (lower R-squared, systematic residual pattern). A low-degree polynomial (degree 2 here, matching how the data was generated) captures the curve and scores clearly higher in cross-validation than the straight line (R-squared about 0.96 vs 0.87), while higher degrees add complexity without improving the cross-validated score and give wiggly, overfitted curves (see the plot). Always select polynomial degree with validation data, not training error.